# SETUP

In [1]:
!pip install ultralytics roboflow wandb --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.6 MB/s eta 0:00:00


In [2]:
from ultralytics import settings
settings.update({'wandb': True})

!yolo settings wandb=True

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Updated 'wandb=True'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/content/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "569f3ba64b326db489132663f79cd37279811de477381b83ac131e6cdd129cbb",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": true,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


# DATASET DOWNLOAD

In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key="qnXeRyr4qHvAKsIEupjE")
project = rf.workspace("joseph-nelson").project("hard-hat-workers")
dataset = project.version(1).download('yolov8', location='/content/dataset')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/dataset in yolov8:: 100%|██████████| 14079/14079 [00:01<00:00, 7147.33it/s]


In [4]:
# Splitting train to make VAL since it wasnt provided as is so

from ultralytics.models.rtdetr import val
import os, shutil, random

train_img_path = '/content/dataset/train/images'
train_label_path = '/content/dataset/train/labels'
val_img_path = '/content/dataset/valid/images'
val_label_path = '/content/dataset/valid/labels'

os.makedirs(val_img_path, exist_ok=True)
os.makedirs(val_label_path, exist_ok=True)

images = os.listdir(train_img_path)
random.seed(42)
val_images = random.sample(images, int(len(images) * 0.2))

for img in val_images:
  shutil.move(f"{train_img_path}/{img}", f"{val_img_path}/{img}")
  lbl = os.path.splitext(img)[0] + '.txt'

  if os.path.exists(f"{train_label_path}/{lbl}"):
    shutil.move(f"{train_label_path}/{lbl}", f"{val_label_path}/{lbl}")

print(f"Train: {len(os.listdir(train_img_path))}")
print(f"Val: {len(os.listdir(val_img_path))}")

Train: 4216
Val: 1053


In [5]:
DATA_YAML = dataset.location + "/data.yaml"
DATA_YAML

'/content/dataset/data.yaml'

In [6]:
import yaml

with open(DATA_YAML) as f:
  data = yaml.safe_load(f)

data["val"] = "valid/images"

with open(DATA_YAML, 'w') as f:
  yaml.dump(data, f)

# TRAINING

In [10]:
from ultralytics import YOLO
import wandb

wandb.login()


weight = YOLO("yolo11s.pt")


results = weight.train(
    # Training params
    data= DATA_YAML,
    epochs= 100,
    batch= 8,
    imgsz= 640,
    device= 0,
    patience= 20,

    # wandb params
    project= "PPE_Saftey_Detection",
    name= "run-3.5",
    exist_ok= True

)

# weight_backup.train(data='/content/dataset/data.yaml', resume=True)

wandb.finish()

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolo_backup.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie


                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256

lr/pg0,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg1,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg2,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
metrics/mAP50(B),▂▄▅▁▄▂▄▁█▃▃▆▆▄▆▅▃▆▅▆▅▆█▄▆▇▇▇█▇▅▆▅▄▄▆▆▅▅▅
metrics/mAP50-95(B),▁▃▄▁▄▃▅▃▅▅▄▆▅▄▇▆▆▆▇▆▆█▇▇▇███▇██▇█████▇██
metrics/precision(B),███████▁██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁
metrics/recall(B),▁▂▁▂▂▂▃▃▂▂▃▄▃▄▃▄▃▃▃▃▄▄▄▃▃█▆▅▆▃▆▅▅▆▆▅▅▆▆▅
train/box_loss,███▇▇█▇▇▇▆▆▇▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▂▂▁
train/cls_loss,██▇▇▇▇▇▇▇▇▆▆▆▆▆▅▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▄▃▂▁▁
train/dfl_loss,███▇▇▇█▇▇▆▆▇▅▅▅▅▅▅▅▄▄▄▄▄▃▃▄▃▃▃▂▂▂▂▂▁▁▃▃▂
+3,...


In [11]:
# Save results to drive
!mkdir -p /content/drive/MyDrive/PPE_FINAL_RESULTS

!cp -r /content/runs/detect/PPE_Saftey_Detection/run-3/weights /content/drive/MyDrive/PPE_FINAL_RESULTS/


✅ MISSION ACCOMPLISHED: Best.pt is safely on your Drive!
